# ML 08h — Machine-Learning Model Comparison

This notebook compares the completed baseline and ML experiments using their saved expanding-window and 2023 validation metrics. It selects a development-stage ML candidate without accessing the sealed 2024 test outcomes.

## Comparison rules

- Every experiment must use the identical split artifact and input hash.
- Primary criterion combines expanding-window mean log-MAE rank and 2023 validation log-MAE rank.
- Baselines are retained as required reference points.
- Kilogram errors, temporal variability and bias are reported alongside the primary metric.
- No training, refitting or final-test scoring occurs here.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','matplotlib':'matplotlib>=3.7,<4'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Comparison dependencies are available.')

In [ ]:
from __future__ import annotations
import hashlib, json, os, sys
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
plt.style.use('seaborn-v0_8-whitegrid')
EXPERIMENTS=['baselines','ridge','random_forest','extra_trees','hist_gradient_boosting']
LEARNED=['ridge','random_forest','extra_trees','hist_gradient_boosting']

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
METRIC_ROOT=ROOT/'outputs'/'metrics'/'model_training'
AUDIT_ROOT=ROOT/'outputs'/'audit'/'model_training'
MODEL_DIR=ROOT/'models'
EVIDENCE_DIR=AUDIT_ROOT/'comparison'; METRIC_DIR=METRIC_ROOT/'comparison'; FIGURE_DIR=ROOT/'outputs'/'figures'/'model_training'/'comparison'
for directory in [EVIDENCE_DIR,METRIC_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
summaries=[]; manifests={}
for experiment in EXPERIMENTS:
    summary_path=METRIC_ROOT/experiment/'experiment_summary.csv'
    manifest_name='baseline_manifest.json' if experiment=='baselines' else 'experiment_manifest.json'
    manifest_path=AUDIT_ROOT/experiment/manifest_name
    if not summary_path.is_file() or not manifest_path.is_file(): raise FileNotFoundError(f'Run the {experiment} notebook first.')
    summaries.append(pd.read_csv(summary_path)); manifests[experiment]=json.loads(manifest_path.read_text(encoding='utf-8'))
comparison=pd.concat(summaries,ignore_index=True)
input_hashes={manifest['input_sha256'] for manifest in manifests.values()}
if len(input_hashes)!=1: raise ValueError('Experiments did not use the same split artifact.')
print('Models:',comparison.model.tolist()); print('Shared input SHA-256:',next(iter(input_hashes)))

In [ ]:
comparison['cv_rank']=comparison.cv_log_mae_mean.rank(method='min')
comparison['validation_rank']=comparison.log_mae.rank(method='min')
comparison['combined_rank']=comparison.cv_rank+comparison.validation_rank
comparison['model_type']=np.where(comparison.model.isin(LEARNED),'learned_model','baseline')
comparison=comparison.sort_values(['combined_rank','log_mae','cv_log_mae_mean']).reset_index(drop=True)
learned=comparison.loc[comparison.model.isin(LEARNED)].copy()
selected=learned.iloc[0]
best_baseline=comparison.loc[comparison.model_type=='baseline'].sort_values('log_mae').iloc[0]
relative_improvement=(best_baseline.log_mae-selected.log_mae)/best_baseline.log_mae*100
selected_model_path=MODEL_DIR/f'{selected.model}_development_pipeline.joblib'
if not selected_model_path.is_file(): raise FileNotFoundError(selected_model_path)
model_sha256=hashlib.sha256(selected_model_path.read_bytes()).hexdigest()
expected_sha256=manifests[str(selected.model)]['model_sha256']
if model_sha256!=expected_sha256: raise ValueError('Selected model hash does not match its experiment manifest.')
comparison.to_csv(METRIC_DIR/'all_model_comparison.csv',index=False)
selection=pd.DataFrame([{'selected_model':selected.model,'selection_rule':'lowest combined CV log-MAE rank plus 2023 validation log-MAE rank among learned models','cv_log_mae_mean':selected.cv_log_mae_mean,'cv_log_mae_std':selected.cv_log_mae_std,'validation_log_mae':selected.log_mae,'validation_kg_mae':selected.kg_mae,'validation_kg_rmse':selected.kg_rmse,'validation_kg_bias':selected.kg_bias,'best_baseline':best_baseline.model,'best_baseline_validation_log_mae':best_baseline.log_mae,'relative_validation_log_mae_improvement_percent':relative_improvement,'candidate_beats_baseline':bool(selected.log_mae<best_baseline.log_mae),'selected_model_file':selected_model_path.name,'selected_model_sha256':model_sha256}])
selection.to_csv(EVIDENCE_DIR/'selected_ml_candidate.csv',index=False)
display(comparison[['model','model_type','cv_log_mae_mean','cv_log_mae_std','log_mae','kg_mae','kg_rmse','kg_bias','combined_rank']]); display(selection)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,6))
order=comparison.sort_values('cv_log_mae_mean')
axes[0].barh(order.model,order.cv_log_mae_mean,xerr=order.cv_log_mae_std,color=['#bab0ac' if kind=='baseline' else '#4e79a7' for kind in order.model_type]); axes[0].set_xlabel('Mean log MAE ± fold SD'); axes[0].set_title('Expanding-window performance')
order=comparison.sort_values('log_mae')
axes[1].barh(order.model,order.log_mae,color=['#bab0ac' if kind=='baseline' else '#f28e2b' for kind in order.model_type]); axes[1].set_xlabel('2023 validation log MAE'); axes[1].set_title('Held-out validation performance')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'ml_model_comparison.png',dpi=200,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(8,7))
for _,row in comparison.iterrows():
    color='#bab0ac' if row.model_type=='baseline' else '#4e79a7'
    ax.scatter(row.cv_log_mae_mean,row.log_mae,s=90,color=color); ax.annotate(row.model,(row.cv_log_mae_mean,row.log_mae),xytext=(5,5),textcoords='offset points')
ax.set_xlabel('Mean expanding-window log MAE'); ax.set_ylabel('2023 validation log MAE'); ax.set_title('Temporal CV versus held-out validation'); fig.tight_layout(); fig.savefig(FIGURE_DIR/'cv_validation_consistency.png',dpi=200,bbox_inches='tight'); plt.show()

In [ ]:
checks=pd.DataFrame([
 {'check':'all required experiments present','passed':set(comparison.model)=={'median_baseline','historical_baseline',*LEARNED},'detail':f'{len(comparison)} results'},
 {'check':'shared split artifact','passed':len(input_hashes)==1,'detail':next(iter(input_hashes))},
 {'check':'all tests sealed','passed':all(not manifest.get('test_target_used',True) for manifest in manifests.values()),'detail':'2024 unused by every experiment'},
 {'check':'selected model is learned','passed':selected.model in LEARNED,'detail':str(selected.model)},
 {'check':'selected model hash verified','passed':model_sha256==expected_sha256,'detail':selected_model_path.name},
 {'check':'finite comparison metrics','passed':np.isfinite(comparison[['cv_log_mae_mean','log_mae','kg_mae']]).all().all(),'detail':'CV and validation metrics'}])
assert checks.passed.all(); checks.to_csv(EVIDENCE_DIR/'comparison_checks.csv',index=False)
findings=pd.DataFrame([
 {'finding':'Selected development ML candidate','result':str(selected.model)},
 {'finding':'Selection method','result':'Combined rank from expanding-window CV and 2023 validation log MAE'},
 {'finding':'Best baseline','result':str(best_baseline.model)},
 {'finding':'Relative validation improvement over best baseline','result':f'{relative_improvement:.2f}%'},
 {'finding':'Candidate beats baseline','result':str(bool(selected.log_mae<best_baseline.log_mae))},
 {'finding':'Final test status','result':'2024 remains sealed; no final performance claim yet'},
 {'finding':'Next stage','result':'Refit selected configuration on train+validation and evaluate 2024 exactly once'}])
findings.to_csv(EVIDENCE_DIR/'comparison_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','experiments':EXPERIMENTS,'shared_input_sha256':next(iter(input_hashes)),'selection_rule':'combined expanding-window CV and 2023 validation log-MAE rank','selected_model':str(selected.model),'selected_model_file':selected_model_path.name,'selected_model_sha256':model_sha256,'candidate_beats_baseline':bool(selected.log_mae<best_baseline.log_mae),'test_target_used':False,'final_test_evaluated':False,'figures':sorted(p.name for p in FIGURE_DIR.glob('*.png')),'metrics':sorted(p.name for p in METRIC_DIR.glob('*.csv'))}
with (EVIDENCE_DIR/'ml_comparison_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(checks); display(findings)

## Viva interpretation

The comparison is fair because every experiment uses the same predictors, chronological folds, validation year and metrics. The selected model is only a development-stage candidate. Its definitive performance must come from one final evaluation on the previously untouched 2024 period.